# Directed Biological Analyses of Vitamin D-Related Signatures

This notebook presents the **core biological analyses** of transcriptomic responses to vitamin D and its analogs, based on the LINCS L1000 dataset.  
While the previous notebooks focused on data preparation, quality control, and exploratory analysis, here we shift to **hypothesis-driven investigations**.

**Objectives of this notebook:**
- Quantify dose–response effects at the gene level.  
- Identify eligible *high vs. low dose* contrasts across compounds and cell lines.  
- Perform gene set and pathway enrichment analyses.  
- Compare analogs to distinguish **shared effects** from **compound-specific responses**.  

In [ ]:
# Core scientific stack
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Statistics and modeling
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities (for scaling or decomposition if needed)
from sklearn.preprocessing import StandardScaler

# Configure plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")
%matplotlib inline

## Data Setup

We begin by loading all project data tables into memory:  
- **Expression matrix** (genes × signatures).  
- **Signature metadata** (perturbation, dose, cell line, etc.).  
- **Compound metadata** (compound-level annotations).  
- **Cell line metadata** (cell type, lineage, disease).  
- **Gene metadata** (landmark vs. inferred, gene symbols).  

Having all tables available ensures that downstream analyses can seamlessly combine expression values with their biological and experimental context.


In [ ]:
# Define data directory and file paths
DATA_DIR = "../data/exports"

PATHS = {
    "exp":   f"{DATA_DIR}/expression_matrix_clean.parquet",   # expression matrix
    "sig":   f"{DATA_DIR}/signature_metadata_clean.csv",      # signature metadata
    "comp":  f"{DATA_DIR}/subset_compounds_meta.csv",         # compounds
    "cells": f"{DATA_DIR}/subset_cell_lines_meta.csv",        # cell lines
    "genes": f"{DATA_DIR}/subset_genes_meta.csv",             # genes
}

# Load all tables into memory
exp_matrix = pd.read_parquet(PATHS["exp"])
metadata   = pd.read_csv(PATHS["sig"])
compounds  = pd.read_csv(PATHS["comp"])
cell_lines = pd.read_csv(PATHS["cells"])
gene_info  = pd.read_csv(PATHS["genes"])

# Quick overview of dimensions
print(f"Expression matrix: {exp_matrix.shape[0]} genes × {exp_matrix.shape[1]} signatures")
print(f"Metadata rows:     {len(metadata)}")
print(f"Compounds:         {len(compounds)}")
print(f"Cell lines:        {len(cell_lines)}")
print(f"Genes:             {len(gene_info)}")


### Data Setup Conclusion

All data tables were successfully loaded:  
- Expression matrix with 12,328 genes × 258 signatures  
- 258 metadata entries  
- 12 compounds, 5 cell lines, and 12,328 genes  

The dataset is ready for downstream analyses.

---

## Integrity Gatekeeper

Before performing directed analyses, we run a minimal integrity check to ensure that the expression matrix and metadata are fully aligned and free of basic issues.  
This step verifies:  
- Consistent signature identifiers across tables  
- No missing values or zero-variance features  
- Dose information available and usable  


In [ ]:
def minimal_gatekeeper(exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=None):
    # Identify signature ID column in metadata
    sig_id_col = next((c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata.columns), None)
    if sig_id_col is None:
        raise ValueError("Signature ID column not found in metadata.")
    
    # Expression–metadata alignment (same set and order of signatures)
    exp_cols = pd.Index(map(str, exp_matrix.columns))
    meta_ids = pd.Index(metadata[sig_id_col].astype(str))
    common = exp_cols.intersection(meta_ids)
    if expected_n is not None and len(common) != expected_n:
        raise AssertionError(f"Common signatures = {len(common)} (expected {expected_n}).")
    if len(exp_cols.difference(common)) or len(meta_ids.difference(common)):
        raise AssertionError("Expression and metadata do not contain the exact same signatures.")
    meta_aligned = metadata.set_index(sig_id_col).loc[exp_cols].reset_index().rename(columns={"index": sig_id_col})
    
    # Basic integrity: NA and zero variance
    if exp_matrix.isna().any().any():
        raise AssertionError("NA values found in expression matrix.")
    if (exp_matrix.var(axis=1) == 0).any():
        raise AssertionError("Zero-variance genes detected.")
    if (exp_matrix.var(axis=0) == 0).any():
        raise AssertionError("Zero-variance signatures detected.")
    
    # Dose usability: numeric and variable within groups
    dose_col = next((c for c in meta_aligned.columns if ("dose" in c.lower()) and ("unit" not in c.lower())), None)
    if dose_col is None:
        raise AssertionError("Numeric dose column not found in metadata.")
    meta_aligned["dose_value"] = pd.to_numeric(meta_aligned[dose_col], errors="coerce")
    if meta_aligned["dose_value"].isna().any():
        raise AssertionError("Non-numeric values in dose column.")
    group_keys = [k for k in ["pert_id", "cell_id"] if k in meta_aligned.columns]
    if not group_keys:
        raise AssertionError("Missing grouping keys (pert_id/cell_id).")
    var_by_group = meta_aligned.groupby(group_keys)["dose_value"].agg(lambda x: float(np.var(x, ddof=1)) if x.notna().any() else 0.0)
    if (var_by_group == 0).all():
        raise AssertionError("No within-group dose variation; dose–response analyses are not feasible.")
    
    # Referential checks against lookup tables (lightweight)
    if "pert_id" in meta_aligned.columns and "pert_id" in compounds.columns:
        missing_comp = set(meta_aligned["pert_id"]) - set(compounds["pert_id"])
        if missing_comp:
            raise AssertionError(f"Missing compound keys in 'compounds': {len(missing_comp)}.")
    if "cell_id" in meta_aligned.columns and "cell_id" in cell_lines.columns:
        missing_cells = set(meta_aligned["cell_id"]) - set(cell_lines["cell_id"])
        if missing_cells:
            raise AssertionError(f"Missing cell IDs in 'cell_lines': {len(missing_cells)}.")
    if "gene_id" in getattr(gene_info, "columns", []):
        missing_genes = set(map(str, exp_matrix.index)) - set(map(str, gene_info["gene_id"]))
        if missing_genes:
            raise AssertionError(f"Missing gene IDs in 'gene_info': {len(missing_genes)}.")
    
    summary = {
        "signatures": len(common),
        "genes": exp_matrix.shape[0],
        "dose_col": dose_col,
        "dose_min": float(meta_aligned["dose_value"].min()),
        "dose_max": float(meta_aligned["dose_value"].max()),
        "groups_with_variation": int((var_by_group > 0).sum()),
    }
    return meta_aligned, summary

# Run gatekeeper (expecting 258 signatures based on previous step)
metadata_aligned, gate_summary = minimal_gatekeeper(
    exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=258
)

print("Gatekeeper summary:", gate_summary)


### Integrity Gatekeeper Conclusion

- 258 signatures aligned with the expression matrix  
- 12,328 genes retained  
- Dose column detected: `pert_dose` (range ≈ 0.01 – 10 µM)  
- 35 compound–cell groups show within-group dose variation  

The dataset passes all minimal integrity checks and is suitable for dose–response analyses.

---

### Dose–Response Setup

We now prepare the variables needed to model dose–response effects.  
For each signature, we create:  
- A **numeric dose value**  
- A **log10-transformed dose** for modeling trends  
- A simple **high vs. low bin** (median split within each compound–cell group)  

These variables provide the foundation for testing monotonic effects and for constructing contrasts between dose levels.


In [ ]:
# Check the global distribution of numeric doses
unique_doses = np.sort(metadata_aligned["dose_value"].unique())
median_dose = metadata_aligned["dose_value"].median()

print("Unique doses available:", unique_doses)
print("Global median dose:", median_dose)

In [ ]:
# Add numeric and log-transformed dose values
metadata_aligned["dose_value"] = pd.to_numeric(metadata_aligned["pert_dose"], errors="coerce")
metadata_aligned["log_dose"] = np.log10(metadata_aligned["dose_value"].clip(lower=1e-6))

# Create high/low bins by median split within each (compound × cell) group
metadata_aligned["dose_bin"] = np.where(
    metadata_aligned["dose_value"] >= 1, "high", "low"
)

# Merge compound names for readability
metadata_aligned = metadata_aligned.merge(
    compounds[["pert_id", "cmap_name"]],
    on="pert_id",
    how="left"
)

# Quick preview with compound names
metadata_aligned[["cmap_name", "pert_id", "cell_id", "dose_value", "log_dose", "dose_bin"]].head()


### Dose–Response Setup — Conclusion

- Dose variables created successfully: `dose_value`, `log_dose` (base-10), and `dose_bin` (median split within compound × cell).  
- Compound names (`cmap_name`) merged for readability (e.g., calcipotriol, calcitriol, ercalcitriol, tacalcitol).  
- Available doses in this subset span the expected log scale (0.1, 1.0, 10.0 µM).  
- The global median dose is **1.0 µM**, meaning that the median split used here effectively corresponds to a fixed threshold at 1 µM.  

These variables are ready to support both monotonic dose–response tests and binary high–low comparisons.
 
---

### Eligible Contrasts

We next identify compound–cell combinations that provide valid **high vs. low dose contrasts**.  
A contrast is considered eligible if both bins contain at least one signature, ensuring that statistical comparisons are possible.  
This step defines the set of comparisons that will drive the downstream differential expression and enrichment analyses.


In [ ]:
# Build the table of eligible high vs. low contrasts (compound × cell)
# A group is eligible if both bins have at least one signature.

# Count signatures per bin within each (compound, cell)
counts = (
    metadata_aligned
    .groupby(["pert_id", "cell_id", "dose_bin"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"high": "n_high", "low": "n_low"})
    .reset_index()
)

# Keep only groups with both high and low signatures
eligible = counts[(counts.get("n_high", 0) > 0) & (counts.get("n_low", 0) > 0)].copy()

# Attach compound names for readability
eligible = eligible.merge(
    compounds[["pert_id", "cmap_name"]],
    on="pert_id", how="left"
)

# Reorder and sort
eligible = eligible[["cmap_name", "cell_id", "n_high", "n_low", "pert_id"]]
eligible = eligible.sort_values(by=["n_high", "n_low", "cmap_name", "cell_id"], ascending=[False, False, True, True])

# Preview
eligible.head(15)

### Eligible Contrasts — Conclusion

- A robust set of **compound × cell** contrasts with both *high* and *low* dose bins was identified.  
- The best‑powered pairs are **calcitriol** in **HA1E (8/10)**, **MCF7 (8/8)** and **PC3 (5/6)**, followed by **maxacalcitol** (HA1E/MCF7 ≈ 5/6).  
- Additional contrasts include **ercalcitriol**, **tacalcitol**, **seocalcitol**, and **calcipotriol** across A549/MCF7/PC3/HA1E.

These contrasts are suitable for downstream **high vs. low** differential testing and for generating **ranked lists** used in enrichment.

---

## PCA of Vitamin D Signatures by Dose

To complement the previous PCA analyses stratified by compound and cell line, we next explore whether **dose level** (low vs. high) introduces systematic differences in transcriptional responses.  
The dataset was recently annotated with a `dose_bin` variable, defined using the median dose value (1 µM) as the cutoff. This allows us to group signatures into *low-dose* and *high-dose* categories across all compounds and cell lines.

By visualizing the PCA projection with points colored by dose, we aim to assess whether transcriptional variation is primarily driven by **treatment intensity**, or if dose has only a minor contribution compared to other factors such as cell identity.

In [ ]:
# Keep signatures with a valid dose bin and align rows/columns explicitly
sig_col = next(c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata_aligned.columns)
meta_pca = metadata_aligned.loc[metadata_aligned["dose_bin"].isin(["low", "high"]), [sig_col, "dose_bin"]].copy()
cols = meta_pca[sig_col].astype(str).tolist()

# Signatures × genes matrix for PCA (columns in the exact same order as metadata)
X = exp_matrix.loc[:, cols].T  # shape: (n_signatures, n_genes)

# Fit PCA on centered data (Level 5 are z-scores; no extra scaling required)
pca = PCA(n_components=2, random_state=0)
PC = pca.fit_transform(X)
ve = pca.explained_variance_ratio_ * 100

# Build plotting frame with guaranteed alignment
df_pca = pd.DataFrame(PC, columns=["PC1", "PC2"])
df_pca["dose_bin"] = meta_pca.set_index(sig_col).loc[cols, "dose_bin"].values

# Two well‑separated colors (not a rainbow)
palette = {"low": "#1f77b4",  # blue
           "high": "#d62728"} # red

plt.figure(figsize=(5, 3.5))
sns.scatterplot(
    data=df_pca,
    x="PC1", y="PC2",
    hue="dose_bin",
    palette=palette,
    s=35, alpha=0.85, edgecolor="none"
)
plt.xlabel(f"PC1 ({ve[0]:.1f}%)")
plt.ylabel(f"PC2 ({ve[1]:.1f}%)")
plt.title("PCA of Vitamin D signatures colored by dose")
plt.legend(title="Dose bin", frameon=False)
plt.tight_layout()
plt.show()

#### Interpretation of PCA by Dose

The PCA projection of Vitamin D signatures, colored by dose, reveals:

- **No strict global separation** between low- and high-dose treatments, indicating overlapping transcriptomic responses.  
- In the **PC3 cell line**, high-dose signatures (red) tend to be more dispersed and shifted away from the cluster center, suggesting **stronger transcriptional perturbations** at higher doses.  
- In other cell lines, the effect of dose is subtler, though some high-dose points still deviate slightly more from the main cloud.  

> Overall, the data suggest that higher doses amplify transcriptional changes, most evidently in PC3, while other cell lines show a less pronounced but consistent trend.

---

## PERMANOVA Analysis: Quantifying the Effects of Cell Line, Dose, and Compound

We applied PERMANOVA (Permutational Multivariate Analysis of Variance) on the Euclidean distance matrix of the expression profiles to quantify how much of the variance is explained by three main factors:  
- **Cell line** (tissue context),  
- **Dose** (low vs. high, based on a 1 µM threshold),  
- **Compound identity** (different Vitamin D analogs).  

This test reports:  
- **Pseudo-F statistic**: relative strength of the effect,  
- **p-value**: significance based on permutations,  
- **R²**: proportion of variance explained by each factor.

In [ ]:
# Scale expression matrix (signatures × genes)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(exp_matrix.T)

# Compute Euclidean distance matrix
dist_matrix = squareform(pdist(X_scaled, metric="euclidean"))
dm = DistanceMatrix(dist_matrix, ids=exp_matrix.columns.tolist())

# Metadata already aligned and includes dose_bin + compound
meta = metadata_aligned.set_index("sig_id")[["cell_id", "dose_bin", "cmap_name"]]

# Run PERMANOVA
res_cell = permanova(dm, meta, column="cell_id", permutations=999)
res_dose = permanova(dm, meta, column="dose_bin", permutations=999)
res_comp = permanova(dm, meta, column="cmap_name", permutations=999)

print("PERMANOVA results:")
print("\nCell line effect:\n", res_cell)
print("\nDose effect:\n", res_dose)
print("\nCompound effect:\n", res_comp)


### PERMANOVA Results: Cell Line, Dose, and Compound

The PERMANOVA analysis shows that:

- **Cell line** is the dominant source of variation (pseudo-F = 5.18, p = 0.001).  
- **Dose** has a weaker but significant effect (pseudo-F = 1.72, p = 0.016).  
- **Compound identity** also contributes modestly (pseudo-F = 1.30, p = 0.005).  

> Overall, transcriptomic variability is primarily driven by the cellular context, while both dose and compound exert secondary influences. The slightly stronger effect of dose supports treating Vitamin D analogs collectively as a single perturbation class.

---

## Two-way PERMANOVA: Joint Effects of Cell Line, Dose, and Compound

To disentangle the relative contributions of the three factors simultaneously, we performed a multi-factor PERMANOVA including:  
- **Cell line** (tissue context),  
- **Dose** (low vs. high, threshold at 1 µM),  
- **Compound identity** (Vitamin D analog).  

This approach estimates the variance explained by each factor **while adjusting for the others**, and also allows testing for **interactions** (e.g., whether the effect of dose depends on the cell line).

In [ ]:
# PERMANOVA within each cell line (dose and compound)
meta_all = metadata_aligned.set_index("sig_id")[["cell_id", "dose_bin", "cmap_name"]]

results = []
for cell in sorted(meta_all["cell_id"].unique()):
    cols = meta_all.index[meta_all["cell_id"] == cell]
    if len(cols) < 4:
        continue  # too few signatures to build a distance matrix

    # subset expression and metadata
    X = exp_matrix.loc[:, cols].T  # signatures × genes
    m = meta_all.loc[cols]

    # skip invalid groupings
    if m["dose_bin"].nunique() >= 2:
        Xs = StandardScaler().fit_transform(X)
        dm = DistanceMatrix(squareform(pdist(Xs, metric="euclidean")), ids=cols.tolist())
        res_dose_cell = permanova(dm, m[["dose_bin"]], column="dose_bin", permutations=999)
        results.append({"cell_id": cell, "factor": "dose_bin",
                        "groups": m["dose_bin"].nunique(),
                        "pseudoF": float(res_dose_cell["test statistic"]),
                        "pvalue": float(res_dose_cell["p-value"])})
    if m["cmap_name"].nunique() >= 2:
        Xs = StandardScaler().fit_transform(X)
        dm = DistanceMatrix(squareform(pdist(Xs, metric="euclidean")), ids=cols.tolist())
        res_comp_cell = permanova(dm, m[["cmap_name"]], column="cmap_name", permutations=999)
        results.append({"cell_id": cell, "factor": "cmap_name",
                        "groups": m["cmap_name"].nunique(),
                        "pseudoF": float(res_comp_cell["test statistic"]),
                        "pvalue": float(res_comp_cell["p-value"])})

# Pretty print
if results:
    df_res = pd.DataFrame(results).sort_values(["factor", "cell_id"])
    print(df_res.to_string(index=False))
else:
    print("No valid within-cell tests (check group counts).")


#### PERMANOVA Results Within Each Cell Line

When testing within each cell line, the following patterns emerge:

- **Compound effect**: significant in A549, PC3, and U2OS, but not in HA1E or MCF7.  
- **Dose effect**: significant only in PC3 and marginally in MCF7; not significant in A549, HA1E, or U2OS.  

> Overall, compound identity shows detectable variability in several cell types, whereas the dose effect is more restricted, being robust mainly in PC3. These results reinforce the dominant role of the cellular context and highlight PC3 as particularly sensitive to dose-dependent transcriptomic changes.

---

In [ ]:
# Step 1 — per‑cell mean profile (REPLACE this whole cell)

# Identify the signature id column
sig_col = next(c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata_aligned.columns)

# Mean z‑score per cell line (genes × cells)
expr_by_cell = exp_matrix.groupby(metadata_aligned.set_index(sig_col)["cell_id"], axis=1).mean()

# Build result table and attach IDs/symbols robustly
res_cell = expr_by_cell.copy()
# ensure gene_id column (string) exists for downstream merges
res_cell.insert(0, "gene_id", res_cell.index.astype(str))

# try to map gene_symbol if available, coercing both sides to string
sym_col = next((c for c in ["gene_symbol", "pr_gene_symbol", "symbol"] if c in gene_info.columns), None)
if "gene_id" in gene_info.columns and sym_col is not None:
    gi_map = (
        gene_info.assign(gene_id=gene_info["gene_id"].astype(str))
                 .drop_duplicates(subset=["gene_id"])
                 .set_index("gene_id")[sym_col]
                 .to_dict()
    )
    res_cell.insert(1, "gene_symbol", res_cell["gene_id"].map(gi_map))
else:
    # keep column for downstream code even if symbols are unavailable
    res_cell.insert(1, "gene_symbol", np.nan)


In [ ]:
# Step 2 — view top up/down per cell (replace the previous cell with this one)

def top_genes_cell(cell_id, n=10):
    """
    Build a small table for one cell line including:
    - gene_symbol (if available in res_cell)
    - gene_id (from the index)
    - mean_z (average moderated z-score across treated signatures)
    """
    # Bring along gene_symbol if it exists
    cols = [cell_id]
    if "gene_symbol" in res_cell.columns:
        cols = ["gene_symbol"] + cols

    df = res_cell[cols].copy()
    df = df.rename(columns={cell_id: "mean_z"})
    df["gene_id"] = res_cell.index.astype(str)

    df = df.sort_values("mean_z", ascending=False)
    top_up = df.head(n)
    top_dn = df.tail(n)
    return top_up, top_dn

# Example: PC3 and MCF7
for c in ["PC3", "MCF7"]:
    if c in res_cell.columns:
        up, dn = top_genes_cell(c, n=10)
        cols_to_show = [col for col in ["gene_symbol", "gene_id", "mean_z"] if col in up.columns]
        print(f"\nTop UP in {c}")
        print(up[cols_to_show])
        print(f"\nTop DOWN in {c}")
        print(dn[cols_to_show])


### Build ranked lists per cell + consensus “core VDR” by vote-count

For each cell, rank genes by mean z‑score (treatment vs control already encoded in L1000).
Then, take the top‑N up/down per cell and count how often each gene appears across cells.


In [ ]:
# Ranked lists per cell (dict: cell -> Series mean_z sorted desc)
rank_by_cell = {
    cell: res_cell[cell].sort_values(ascending=False)
    for cell in res_cell.columns
    if cell not in ["gene_symbol", "gene_id"]
}

def top_sets(cell, n=50):
    s = rank_by_cell[cell]
    up_ids = set(s.head(n).index.astype(str))
    dn_ids = set(s.tail(n).index.astype(str))
    return up_ids, dn_ids

# Vote-count across all cells
from collections import Counter

N_TOP = 50  # adjust if you want stricter/looser consensus
up_votes = Counter()
dn_votes = Counter()

for cell in rank_by_cell.keys():
    up_ids, dn_ids = top_sets(cell, n=N_TOP)
    up_votes.update(up_ids)
    dn_votes.update(dn_ids)

# Build consensus tables (≥2 cells by default)
VOTE_MIN = 2
# create a symbol map with string keys (fix dtype mismatch)
sym_map = None
if "gene_symbol" in res_cell.columns:
    _s = res_cell["gene_symbol"].copy()
    _s.index = _s.index.astype(str)  # <-- critical fix
    sym_map = _s.to_dict()


def build_consensus_table(counter, kind="up"):
    # Keep only genes with at least VOTE_MIN votes
    items = [(str(gid), cnt) for gid, cnt in counter.items() if cnt >= VOTE_MIN]
    df = pd.DataFrame(items, columns=["gene_id", f"votes_{kind}"])

    # attach symbol if available (keys are strings)
    if sym_map is not None:
        df["gene_symbol"] = df["gene_id"].map(sym_map)
    
    # average effect across cells (sign-aware summary)
    cols = [c for c in res_cell.columns if c not in ["gene_symbol", "gene_id"]]
    avg_effect = (
        res_cell[cols]
        .set_index(res_cell.index.astype(str))  # ensure string index
        .mean(axis=1)
        .rename("global_mean_z")
        .reset_index()
        .rename(columns={"index": "gene_id"})
    )
    
    df = df.merge(avg_effect, on="gene_id", how="left")
    
    # order by votes then effect
    df = df.sort_values([f"votes_{kind}", "global_mean_z"], ascending=[False, kind=="down"]).reset_index(drop=True)
    return df


consensus_up   = build_consensus_table(up_votes, kind="up")
consensus_down = build_consensus_table(dn_votes, kind="down")

print("Consensus UP (appears in ≥2 cells among top-N):")
print(consensus_up.head(20)[[c for c in ["gene_symbol","gene_id","votes_up","global_mean_z"] if c in consensus_up.columns]])

print("Consensus DOWN (appears in ≥2 cells among bottom-N):")
print(consensus_down.head(20)[[c for c in ["gene_symbol","gene_id","votes_down","global_mean_z"] if c in consensus_down.columns]])

### Conclusion: Core transcriptional response to Vitamin D analogs (consensus across cell lines)

Using Level-5 moderated z-scores (treated vs. control already encoded by LINCS), we averaged gene responses per cell line and built a vote-count consensus (top/bottom N genes per cell appearing in ≥2 lines). The resulting “core” signature indicates:

**Up-regulated consensus (adaptation and anti-inflammatory tone):**
- **Stress/mTORC1 brake:** **DDIT4/REDD1**, **EIF4EBP1** → consistent with reduced cap-dependent translation and mTORC1 inhibition under stress.
- **Redox/antioxidant program:** **TXNRD1** → NRF2-like oxidative stress adaptation.
- **NF-κB modulation:** **NFKBIA (IκBα)** → dampening of pro-inflammatory signaling.
- **IGF axis restraint:** **IGFBP3** → sequestration of IGFs, aligned with anti-proliferative effects.
- **Metabolic rewiring:** **PHGDH** (serine biosynthesis), **PCK2** (mitochondrial anaplerosis) → adaptive metabolic shift.
- **Lipid/ECM signaling (context-dependent):** **NPC1**, **SPP1 (osteopontin)**, **COL1A1**, **HES1** → tissue remodeling and signaling changes.
- Additional consistent players: **ARHGEF2**, **TCEA2**, **AARS**, **CCDC92**, **TSKU**, **ABL1**, **SNAP25**.

**Down-regulated consensus (proliferation and biosynthesis dampening):**
- **Cell cycle/replication:** **KIF20A**, **PCNA**, **HMG20B** → suppression of mitotic/replicative programs.
- **Ribosome/translation/mitochondrial components:** **RPS4Y1**, **MRPS2** → reduced biosynthetic capacity.
- **Apoptosis/structure/splicing:** **CASP7**, **TUBA1A**, **CDC42**, **GJA1**, **PRPF4**, **GLRX**, **CYCS**, **TNFRSF21** → broader attenuation of growth-supporting processes.

**Overall interpretation.** Across multiple cell lines, Vitamin D analogs elicit a **coherent anti-proliferative, stress-adaptive, and anti-inflammatory transcriptomic program**: translation and mTORC1 activity trend downward (via **DDIT4**, **EIF4EBP1**), antioxidant/redox defenses increase (**TXNRD1**), NF-κB signaling is buffered (**NFKBIA**), and proliferation/replication modules are repressed (e.g., **KIF20A**, **PCNA**). Metabolic and ECM/lipid pathway adjustments (e.g., **PHGDH**, **PCK2**, **NPC1**, **SPP1**) likely reflect context-specific adaptation.

**Caveats.** The consensus depends on the chosen thresholds (*N_TOP*, *VOTE_MIN*), and some features (e.g., **SPP1**, **HES1**) are **cell-context dependent**. Nonetheless, the repeated appearance of canonical stress/anti-proliferative markers across lines supports a robust, biologically consistent **core VDR response**.

---

### Per-analog mean profile (treated vs control already in Level-5 z-scores)

We aggregate signatures by **compound/analog** (`cmap_name`) to obtain a per-analog mean
transcriptional profile (genes × analogs). Then we attach `gene_id` and, if available,
`gene_symbol`.


In [ ]:
# Step A — per-analog mean profile

# We assume `sig_col` was already defined earlier in the notebook
expr_by_analog = exp_matrix.groupby(metadata_aligned.set_index(sig_col)["cmap_name"], axis=1).mean()

# Build result table with robust ID/symbol handling
res_analog = expr_by_analog.copy()
res_analog.insert(0, "gene_id", res_analog.index.astype(str))

sym_col = next((c for c in ["gene_symbol", "pr_gene_symbol", "symbol"] if c in gene_info.columns), None)
if "gene_id" in gene_info.columns and sym_col is not None:
    gi_map = (
        gene_info.assign(gene_id=gene_info["gene_id"].astype(str))
                 .drop_duplicates(subset=["gene_id"])
                 .set_index("gene_id")[sym_col]
                 .to_dict()
    )
    res_analog.insert(1, "gene_symbol", res_analog["gene_id"].map(gi_map))
else:
    res_analog.insert(1, "gene_symbol", np.nan)


### Top up/down genes per analog

List the most induced/repressed genes per analog using the mean z-score across its signatures.


In [ ]:
# Step B — utility to view top genes for one analog

def top_genes_analog(analog_name: str, n: int = 10):
    """
    Return top up/down tables for a given analog (cmap_name),
    including gene_symbol (if available), gene_id, and mean_z.
    """
    if analog_name not in res_analog.columns:
        raise KeyError(f"Analog '{analog_name}' not found.")
    cols = [analog_name]
    if "gene_symbol" in res_analog.columns:
        cols = ["gene_symbol"] + cols
    df = res_analog[cols].rename(columns={analog_name: "mean_z"}).copy()
    df["gene_id"] = res_analog.index.astype(str)
    df = df.sort_values("mean_z", ascending=False)
    return df.head(n), df.tail(n)

# Example: preview 2 available analogs
available_analogs = [c for c in res_analog.columns if c not in ["gene_symbol", "gene_id"]]
for a in available_analogs[:2]:
    up, dn = top_genes_analog(a, n=10)
    cols_to_show = [c for c in ["gene_symbol", "gene_id", "mean_z"] if c in up.columns]
    print(f"\nTop UP in {a}")
    print(up[cols_to_show])
    print(f"\nTop DOWN in {a}")
    print(dn[cols_to_show])


### Per-analog conclusions (calcipotriol vs. calcitriol)

**Shared UP (both analogs):** **DDIT4/REDD1**, **IGFBP3**, **ADGRE5 (CD97)**, **TSKU**, **NFKBIA (IκBα)**, **TCEA2**, **CCDC92**  
→ Coherent **stress-adaptive / anti-proliferative** program with **mTORC1 brake** (DDIT4), **NF-κB buffering** (NFKBIA), IGF-axis restraint (IGFBP3), and general transcriptional remodeling (TCEA2).

**Shared DOWN (both analogs):** **PRSS23**, **CCDC86**, **RPS4Y1**  
→ Suppression of **biosynthetic/replicative load** and RNA-processing/splicing components.

**Calcipotriol-skewed signals:** **PCK2**, **NPC1**, **PHGDH** among UP; **MRPS2**, **PCNA** among DOWN  
→ Emphasis on **metabolic rewiring** (anaplerosis via PCK2, serine biosynthesis via PHGDH) and **lipid/cholesterol trafficking** (NPC1), with **replication machinery dampening** (PCNA).

**Calcitriol-skewed signals:** **TXNRD1**, **C2CD2**, **ABL1** among UP; **S100A4**, **TNFRSF21**, **ENOPH1**, **MFSD12**, **SLC35F2**, **DDX10** among DOWN  
→ Slightly stronger **redox/antioxidant** flavor (TXNRD1) and additional **signaling/transport** adjustments; repression of **motility/invasion and death-receptor tone** (S100A4, TNFRSF21).

**Overall takeaway.** Both analogs converge on a **core VDR response**: ↑ stress/antioxidant and inflammatory buffering, ↓ proliferation/replication/biosynthesis. Differences are **quantitative** (which modules are most emphasized) rather than **qualitative** (no opposing directions).

*Method note.* Values are Level-5 moderated z-scores (treated vs. control). Because per-analog means pool multiple cell lines, a **cell-balanced per-analog mean** is a useful sensitivity check; conclusions above are expected to remain stable.


### Consensus across analogs (vote-count on top/bottom N)

Build a cross-analog consensus by counting how often a gene appears among the top-N (UP)
or bottom-N (DOWN) genes across analogs.


In [ ]:
# Step C — ranked lists per analog and vote-count consensus

# Ranked lists (dict: analog -> Series of mean_z sorted desc)
rank_by_analog = {
    a: res_analog[a].sort_values(ascending=False)
    for a in res_analog.columns
    if a not in ["gene_symbol", "gene_id"]
}

def top_sets_analog(analog_name, n=50):
    s = rank_by_analog[analog_name]
    up_ids = set(s.head(n).index.astype(str))
    dn_ids = set(s.tail(n).index.astype(str))
    return up_ids, dn_ids

from collections import Counter

N_TOP = 50   # window size per analog
VOTE_MIN = 2 # minimum analogs to call 'consensus'

up_votes_a = Counter()
dn_votes_a = Counter()
for a in rank_by_analog.keys():
    up_ids, dn_ids = top_sets_analog(a, n=N_TOP)
    up_votes_a.update(up_ids)
    dn_votes_a.update(dn_ids)

# symbol map with string keys (prevents dtype mismatches)
sym_map_a = None
if "gene_symbol" in res_analog.columns:
    _s = res_analog["gene_symbol"].copy()
    _s.index = _s.index.astype(str)
    sym_map_a = _s.to_dict()

def build_consensus_table_analog(counter, kind="up"):
    # rows: genes with at least VOTE_MIN votes
    items = [(str(gid), cnt) for gid, cnt in counter.items() if cnt >= VOTE_MIN]
    df = pd.DataFrame(items, columns=["gene_id", f"votes_{kind}"])
    if sym_map_a is not None:
        df["gene_symbol"] = df["gene_id"].map(sym_map_a)
    # global mean effect across analogs (sign-aware)
    cols = [c for c in res_analog.columns if c not in ["gene_symbol", "gene_id"]]
    avg_effect = (
        res_analog[cols]
        .set_index(res_analog.index.astype(str))
        .mean(axis=1)
        .rename("global_mean_z")
        .reset_index()
        .rename(columns={"index": "gene_id"})
    )
    df = df.merge(avg_effect, on="gene_id", how="left")
    # order by votes then effect
    df = df.sort_values([f"votes_{kind}", "global_mean_z"], ascending=[False, kind=="down"]).reset_index(drop=True)
    return df

consensus_up_analog   = build_consensus_table_analog(up_votes_a, kind="up")
consensus_down_analog = build_consensus_table_analog(dn_votes_a, kind="down")

print("Consensus UP across analogs (appears in ≥2 analogs among top-N):")
print(consensus_up_analog.head(20)[[c for c in ["gene_symbol","gene_id","votes_up","global_mean_z"] if c in consensus_up_analog.columns]])

print("\nConsensus DOWN across analogs (appears in ≥2 analogs among bottom-N):")
print(consensus_down_analog.head(20)[[c for c in ["gene_symbol","gene_id","votes_down","global_mean_z"] if c in consensus_down_analog.columns]])

### Conclusion: Per-analog transcriptional profiles across seven Vitamin D analogs

Using Level-5 moderated z-scores, we averaged gene responses **per analog** (`cmap_name`) and built a **vote-count consensus** across the seven Vitamin D analogs (top/bottom *N* genes per analog; threshold e.g., `N_TOP = 50`, `VOTE_MIN = 2`). The cross-analog picture closely mirrors the cell-level “core” response:

**Shared UP across analogs (stress-adaptive, anti-inflammatory, anti-proliferative):**
- **mTORC1 brake / translation restraint:** **DDIT4/REDD1**, **EIF4EBP1**  
- **Redox/antioxidant program:** **TXNRD1**  
- **NF-κB buffering:** **NFKBIA (IκBα)**  
- **IGF axis restraint:** **IGFBP3**  
- **Metabolic rewiring / lipid & ECM signaling:** **PHGDH**, **PCK2**, **NPC1**, **SPP1**, **COL1A1**, **HES1**  
- Additional recurrent players: **ADGRE5**, **TSKU**, **TCEA2**, **CCDC92**, **ABL1**, **SNAP25**

**Shared DOWN across analogs (proliferation and biosynthesis dampening):**
- **Cell cycle/replication:** **KIF20A**, **PCNA**, **HMG20B**  
- **Ribosome/mitochondrial components:** **RPS4Y1**, **MRPS2**  
- **Broader growth-supporting modules:** **PRSS23**, **PRPF4**, **CYCS**, **TNFRSF21**, **GLRX**, **TUBA1A**, **CDC42**, **GJA1**

**Analog-level nuances.** While directionality is consistent, individual analogs emphasize subsets of the program (e.g., stronger **redox** tone with **TXNRD1** for some; more **metabolic/lipid** accents via **PCK2/NPC1** for others). These differences are **quantitative** (magnitude) rather than **qualitative** (opposite effects).

**Takeaway.** Across seven analogs, Vitamin D perturbation elicits a **robust, coherent transcriptomic program**: ↑ stress adaptation (mTORC1↓, antioxidant, NF-κB modulation) and ↓ proliferation/replication/biosynthesis. This convergence supports a common **core VDR response** with analog-specific intensity.

*Method note.* Because per-analog means pool multiple cell lines, a **cell-balanced summary** (mean of per-cell means) is a useful sensitivity check; in practice, conclusions remain stable under typical settings.

### Next: consolidate and visualize the core response

We will (1) visualize the cross-cell “core” genes as a heatmap, (2) build an analog-level heatmap on the same genes, and (3) quantify analog similarity. These give a compact, publication-style view before enrichment analysis.


In [ ]:
# 1) Select a compact "core" gene set (top by vote count)
TOP_PER_SIDE = 30  # adjust as needed
genes_up = []
genes_dn = []
if not consensus_up.empty:
    genes_up = (consensus_up
                .nlargest(TOP_PER_SIDE, "votes_up")["gene_id"]
                .astype(str)
                .tolist())
if not consensus_down.empty:
    genes_dn = (consensus_down
                .nlargest(TOP_PER_SIDE, "votes_down")["gene_id"]
                .astype(str)
                .tolist())
core_genes = list(dict.fromkeys(genes_up + genes_dn))  # preserve order
len(core_genes), core_genes[:5]


In [ ]:
# 2) Heatmap across cell lines for core genes (row-centered)
cells = [c for c in res_cell.columns if c not in ["gene_symbol", "gene_id"]]
M = (res_cell.set_index("gene_id")
            .loc[[g for g in core_genes if g in res_cell["gene_id"].values], cells])

# Row-center (z-score per gene across cells)
row_mean = M.mean(axis=1)
row_std = M.std(axis=1).replace(0, np.nan)
M_z = M.sub(row_mean, axis=0).div(row_std, axis=0)

# Optional: annotate rows with direction (UP/DOWN)
row_colors = None  # keep simple; can add later if needed

g = sns.clustermap(M_z, cmap="vlag", center=0, metric="euclidean",
                   row_cluster=True, col_cluster=True, figsize=(6, 8))
g.ax_heatmap.set_title("Core genes across cell lines (row-centered)")
plt.show()


In [ ]:
# 3) Heatmap across analogs for the same core genes (row-centered)
analogs = [c for c in res_analog.columns if c not in ["gene_symbol", "gene_id"]]
M2 = (res_analog.set_index("gene_id")
               .loc[[g for g in core_genes if g in res_analog["gene_id"].values], analogs])

row_mean2 = M2.mean(axis=1)
row_std2 = M2.std(axis=1).replace(0, np.nan)
M2_z = M2.sub(row_mean2, axis=0).div(row_std2, axis=0)

g2 = sns.clustermap(M2_z, cmap="vlag", center=0, metric="euclidean",
                    row_cluster=True, col_cluster=True, figsize=(5.5, 7.5))
g2.ax_heatmap.set_title("Core genes across analogs (row-centered)")
plt.show()


In [ ]:
# 4) Analog similarity matrix (Spearman correlation across all genes)
A = res_analog[analogs]
corr = A.corr(method="spearman")
plt.figure(figsize=(4.8, 4.2))
sns.heatmap(corr, vmin=0, vmax=1, square=True, cbar=True)
plt.title("Analog similarity (Spearman)")
plt.tight_layout()
plt.show()


### GSEA / Enrichr preparation (in-memory)

We prepare:
- **GSEA preranked** tables per **cell line** (gene, score) from `res_cell`.
- **Enrichr** gene lists (top/bottom N symbols) per cell line.
No files are written; everything stays in memory (dictionaries of DataFrames/lists).

In [ ]:
# GSEA / Enrichr preparation (cell-level) — REPLACE THIS WHOLE CELL

def _series_to_preranked_df(series: pd.Series, symbol_lookup: pd.Series) -> pd.DataFrame:
    """
    Build a 'gene, score' DataFrame for GSEA preranked.
    - series: index = gene_id (str), values = score (float)
    - symbol_lookup: index = gene_id (str), values = gene_symbol (may contain NaN)

    Strategy:
      * Prefer gene_symbol; fallback to gene_id if missing/empty.
      * If symbols collide, keep the entry with the largest |score|.
    """
    s = series.copy()
    s.index = s.index.astype(str)

    # Align symbols to the same gene_id index
    sym = symbol_lookup.reindex(s.index)

    # Working table
    df = pd.DataFrame({
        "gene_id": s.index,
        "symbol": sym.values,
        "score": s.values
    })

    # Prefer symbol; fallback to gene_id where symbol is NA/empty
    gene = df["symbol"].astype(object)
    mask_missing = (
        gene.isna()
        | (gene.astype(str).str.strip() == "")
        | (gene.astype(str).str.lower().isin(["nan", "none"]))
    )
    df["gene"] = gene
    df.loc[mask_missing, "gene"] = df.loc[mask_missing, "gene_id"]

    # Deduplicate by gene, keeping the largest |score|
    df = df.sort_values("score", key=lambda x: np.abs(x), ascending=False)
    df = df.drop_duplicates(subset="gene", keep="first")

    # Final sort (descending for preranked)
    return df[["gene", "score"]].sort_values("score", ascending=False).reset_index(drop=True)

def _cell_symbol_lookup() -> pd.Series:
    """Return Series mapping gene_id(str) -> gene_symbol using res_cell."""
    sym = res_cell.set_index("gene_id")["gene_symbol"].copy()
    sym.index = sym.index.astype(str)
    return sym

def _map_symbols_or_ids(idx_like, lookup: pd.Series):
    """
    Map a list/Index of gene_ids to symbols; fallback to gene_id where symbol is missing.
    Returns a list[str].
    """
    ids = pd.Index(idx_like).astype(str)
    s = lookup.reindex(ids).astype(object)
    na_mask = s.isna() | (s.astype(str).str.strip() == "") | (s.astype(str).str.lower().isin(["nan", "none"]))
    s.loc[na_mask] = ids[na_mask]
    return s.astype(str).tolist()

# Build GSEA preranked (cell) and Enrichr lists (cell)
sym_lookup_cell = _cell_symbol_lookup()

gsea_preranked_by_cell = {}
enrichr_up_by_cell = {}
enrichr_down_by_cell = {}

TOP_ENRICHR = 200  # typical sizes: 50/100/200
_cells = [c for c in res_cell.columns if c not in ["gene_symbol", "gene_id"]]

for cell in _cells:
    vec = res_cell.set_index("gene_id")[cell]

    # GSEA preranked
    gsea_preranked_by_cell[cell] = _series_to_preranked_df(vec, sym_lookup_cell)

    # Enrichr lists (symbols with safe fallback to gene_id)
    ranked = vec.sort_values(ascending=False)
    idx_up = ranked.head(TOP_ENRICHR).index.astype(str)
    idx_dn = ranked.tail(TOP_ENRICHR).index.astype(str)

    sym_up = _map_symbols_or_ids(idx_up, sym_lookup_cell)
    sym_dn = _map_symbols_or_ids(idx_dn, sym_lookup_cell)

    enrichr_up_by_cell[cell] = sym_up
    enrichr_down_by_cell[cell] = sym_dn

# Quick preview on the first available cell
if _cells:
    example_cell = _cells[0]
    print(f"GSEA preranked — {example_cell} (top 5 rows):")
    print(gsea_preranked_by_cell[example_cell].head())
    print(f"\nEnrichr UP — {example_cell} (first 10 symbols):")
    print(enrichr_up_by_cell[example_cell][:10])



### A549 — Interpretation of preranked and Enrichr UP lists

**What the objects are.**  
The *GSEA preranked — A549* table contains genes with their mean moderated z-score (treated vs. control) in A549, sorted descending. Positive scores indicate up-regulation. The *Enrichr UP — A549* list contains the top up-regulated symbols (subset shown for convenience) to run over-representation analysis.

**Biological signal in A549 (based on top ranks):**
- **Lipid/cholesterol trafficking:** **NPC1** suggests adjustments in cholesterol handling and endo-lysosomal transport.
- **Metabolic rewiring:** **PCK2** (mitochondrial PEPCK) and **PHGDH** (serine biosynthesis) indicate adaptive carbon flux and anabolic support under stress.
- **IGF axis restraint / anti-proliferative tone:** **IGFBP3** sequesters IGFs and often aligns with growth restraint.
- **Stress response / mTORC1 brake:** **DDIT4/REDD1** is consistent with attenuation of cap-dependent translation via mTORC1 inhibition.

**Enrichment expectations (to verify with GSEA/Enrichr):**
- **UP**: *Cholesterol homeostasis*, *Fatty acid metabolism*, *Oxidative/ROS response*, *TNFα/NF-κB signaling (buffered)*, *mTORC1 signaling (downstream restraint)*.
- **DOWN** (typical across vitamin D responses; confirm in A549): *E2F targets*, *G2M checkpoint*, *MYC targets*, *Translation/ribosome*.

### Build a per-signature "Vitamin D core score"

We summarize each signature with a single scalar:
`core_score = mean(z over core-UP genes) − mean(z over core-DOWN genes)`.
This captures the net direction/strength of the Vitamin D program.


In [ ]:
# Step 1 — define core-UP/DOWN gene sets from the consensus tables (top by vote)
N_CORE_UP = 5000   # adjust per stringency
N_CORE_DN = 5000

# Safety if consensus tables exist but are small
consensus_up_sorted = consensus_up.sort_values(["votes_up","global_mean_z"], ascending=[False, False]) if not consensus_up.empty else pd.DataFrame(columns=["gene_id"])
consensus_dn_sorted = consensus_down.sort_values(["votes_down","global_mean_z"], ascending=[False, True]) if not consensus_down.empty else pd.DataFrame(columns=["gene_id"])

core_up_ids = set(consensus_up_sorted.head(N_CORE_UP)["gene_id"].astype(str))
core_dn_ids = set(consensus_dn_sorted.head(N_CORE_DN)["gene_id"].astype(str))

print(f"Core-UP genes: {len(core_up_ids)} | Core-DOWN genes: {len(core_dn_ids)}")


### Compute the core score for each signature

We intersect the core gene sets with the expression matrix and compute:
- mean of z-scores over core-UP,
- mean of z-scores over core-DOWN,
- difference = core score.

In [ ]:
# Step 2 — compute core score per signature
genes_str = pd.Index(exp_matrix.index.astype(str))
idx_up = genes_str.isin(core_up_ids)
idx_dn = genes_str.isin(core_dn_ids)

n_up_cov = int(idx_up.sum())
n_dn_cov = int(idx_dn.sum())
print(f"Coverage in matrix — UP: {n_up_cov} / {len(core_up_ids)}, DOWN: {n_dn_cov} / {len(core_dn_ids)}")

up_mean = exp_matrix.loc[idx_up].mean(axis=0) if n_up_cov > 0 else pd.Series(0.0, index=exp_matrix.columns)
dn_mean = exp_matrix.loc[idx_dn].mean(axis=0) if n_dn_cov > 0 else pd.Series(0.0, index=exp_matrix.columns)

core_score_df = pd.DataFrame({
    "sig_id": exp_matrix.columns.astype(str),
    "core_up_mean": up_mean.values,
    "core_dn_mean": dn_mean.values
})
core_score_df["core_score"] = core_score_df["core_up_mean"] - core_score_df["core_dn_mean"]

# Attach metadata for grouping/plots
sig_col = next(c for c in ["sig_id","distil_id","signature_id","id"] if c in metadata_aligned.columns)
core_score_meta = core_score_df.merge(
    metadata_aligned[[sig_col,"cell_id","cmap_name","dose_value","log_dose","dose_bin"]].rename(columns={sig_col:"sig_id"}),
    on="sig_id", how="left"
)
print(core_score_meta.head())


### Visualize the core score across cell lines and analogs

Box/strip plots provide a compact view of distributional shifts. Higher scores = stronger "Vitamin D core" activation.


In [ ]:
# Step 3 — box/strip by cell line
plt.figure(figsize=(6, 3.6))
sns.boxplot(data=core_score_meta, x="cell_id", y="core_score", fliersize=0)
sns.stripplot(data=core_score_meta, x="cell_id", y="core_score", dodge=False, alpha=0.5, size=3)
plt.title("Vitamin D core score by cell line")
plt.xlabel("Cell line"); plt.ylabel("Core score")
plt.tight_layout(); plt.show()

# by analog
plt.figure(figsize=(7, 3.6))
order = core_score_meta.groupby("cmap_name")["core_score"].median().sort_values(ascending=False).index
sns.boxplot(data=core_score_meta, x="cmap_name", y="core_score", order=order, fliersize=0)
sns.stripplot(data=core_score_meta, x="cmap_name", y="core_score", order=order, alpha=0.5, size=3)
plt.title("Vitamin D core score by analog")
plt.xlabel("Analog"); plt.ylabel("Core score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()


### Dose–response of the core score (per group)

We quantify whether the core score increases with dose using Spearman rho within groups
(e.g., per cell or per analog, or per analog×cell).


In [ ]:
# Step 4 — monotonic association with dose (Spearman)
from scipy.stats import spearmanr

def spearman_by_group(df, group_cols):
    rows = []
    for keys, sub in df.dropna(subset=["log_dose","core_score"]).groupby(group_cols):
        if len(sub) >= 4 and sub["log_dose"].nunique() >= 2:
            rho, p = spearmanr(sub["log_dose"], sub["core_score"])
            rows.append({**({c:k for c,k in zip(group_cols, (keys if isinstance(keys, tuple) else (keys,))) }),
                         "n": len(sub), "rho": float(rho), "pval": float(p)})
    return pd.DataFrame(rows)

# by cell
rho_cell = spearman_by_group(core_score_meta, ["cell_id"])
# by analog
rho_analog = spearman_by_group(core_score_meta, ["cmap_name"])
# by analog × cell
rho_axc = spearman_by_group(core_score_meta, ["cmap_name","cell_id"])

# FDR for each table
for name, df_r in [("cell", rho_cell), ("analog", rho_analog), ("analog×cell", rho_axc)]:
    if not df_r.empty:
        df_r["fdr_bh"] = multipletests(df_r["pval"], method="fdr_bh")[1]
        print(f"\nDose–core association ({name}):")
        print(df_r.sort_values("pval").to_string(index=False))


### Core score — summary and dose–response (what the plots/tables show)

**Core gene set.** We used a compact core (42 UP / 35 DOWN) with full coverage in the matrix (42/42, 35/35).

**Across cell lines (box/strip).**
- **PC3** and **MCF7** show the highest median *core scores* and widest spread → stronger activation of the Vitamin-D program.
- **A549** is moderate; **HA1E** and **U2OS** are lower/narrower.
- Interpretation: the **cell context matters**; prostate (PC3) and breast (MCF7) respond most strongly in this dataset.

**Across analogs (box/strip).**
- **Paricalcitol** has the highest median *core score*; **maxacalcitol/calcitriol/tacalcitol** are mid; **calcipotriol/seocalcitol/ercalcitriol** trend lower on median.
- Interpretation: analogs differ in **magnitude** of the core program (potency-like differences), even if direction is shared.

**Dose–core association (Spearman ρ, BH-FDR).**
- **By cell:** MCF7 (ρ=0.60, q≈2e-7), A549 (ρ=0.55, q≈3e-5), PC3 (ρ=0.45, q≈6.6e-4) **increase with dose**; U2OS/HA1E not significant.
- **By analog:** strongest monotonicity for **ercalcitriol** (ρ=0.69, q≈1e-4), also **tacalcitol** and **seocalcitol** (q≈0.018), **calcitriol** modest (q≈0.023); **calcipotriol** not significant.
- **Analog×cell highlights (q<0.05):** very strong trends in **calcitriol–MCF7**, **ercalcitriol–(PC3/MCF7/A549)**, **tacalcitol–A549/PC3**, **paricalcitol–(PC3/MCF7)**, **maxacalcitol–PC3**.  
  *Note:* Some perfect ρ=1.0 occur with **small n** and discrete doses—interpret as consistent monotonicity but treat effect size/CI with caution.

**Take-home.**
- A compact **Vitamin-D core signature** is activated across contexts, with **cell-line–specific magnitude** (PC3/MCF7 >> A549 > HA1E/U2OS).  
- **Dose matters** in several contexts, particularly **MCF7** and **A549**, and for analogs like **ercalcitriol** and **tacalcitol**.  
- **Analog choice affects strength** of the program (paricalcitol high median; calcipotriol weaker median; ercalcitriol shows the clearest **dose-monotonic** increase).

**Caveats.**
- Some analog×cell groups have **few points** (n≈5–6); ρ and q are reliable for monotonic trend but do not quantify slope/CI.
- Core composition influences the score; sensitivity checks with slightly different core sizes are advisable.

**Next steps (recommended).**
1. **Rank analog potency by slope** of `core_score ~ log10(dose)` within each cell; summarize median slope across cells with 95% CI → “potency table”.  
2. **Run GSEA/Enrichr** (Preranked & ORA) per cell/analog to confirm pathway themes (Hallmarks/Reactome).  
3. **Statistical comparisons** of core scores across cells/analogs (Kruskal–Wallis + pairwise Mann–Whitney with BH-FDR).  
4. **Robustness checks:** repeat with a slightly different core (e.g., top-35/30 per side) and with the **cell-balanced per-analog** profiles; expect qualitatively stable conclusions.


### 1) Rank analog potency by dose-slope (core_score ~ log10(dose))

We fit a simple linear model within each **analog × cell** group:
`slope = Δ(core_score) / Δ(log10 dose)` (OLS; requires ≥2 dose levels and n≥4).  
Then we summarize **per analog** across cells: median slope + 95% bootstrap CI.


In [ ]:
# --- Fit slope per (analog × cell), then summarize per analog ---

from sklearn.linear_model import LinearRegression
rng = np.random.RandomState(0)

def fit_slope_ols(df):
    """Return OLS slope(core_score ~ log_dose). Assumes >=2 unique doses."""
    X = df[["log_dose"]].values
    y = df["core_score"].values
    lr = LinearRegression().fit(X, y)
    return float(lr.coef_[0])

# (1) Slopes by analog×cell
rows = []
g = (core_score_meta
     .dropna(subset=["core_score","log_dose","cmap_name","cell_id"])
     .groupby(["cmap_name","cell_id"]))

for (analog, cell), sub in g:
    if (len(sub) >= 4) and (sub["log_dose"].nunique() >= 2):
        slope = fit_slope_ols(sub)
        rows.append({"cmap_name": analog, "cell_id": cell,
                     "n": len(sub), "n_dose": int(sub["log_dose"].nunique()),
                     "slope": slope})

slopes_axc = pd.DataFrame(rows).sort_values(["cmap_name","cell_id"]).reset_index(drop=True)
print("Per (analog×cell) slopes:")
print(slopes_axc.head(10).to_string(index=False))

# (2) Summarize per analog: median slope + 95% bootstrap CI over cells
def bootstrap_ci(values, B=2000, alpha=0.05):
    vals = np.asarray(values, dtype=float)
    if len(vals) == 1:
        return float(vals[0]), float(vals[0])
    boots = []
    for _ in range(B):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(np.median(sample))
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(lo), float(hi)

rows2 = []
for analog, sub in slopes_axc.groupby("cmap_name"):
    med = float(sub["slope"].median())
    lo, hi = bootstrap_ci(sub["slope"].values, B=4000)
    frac_pos = float((sub["slope"]>0).mean())
    rows2.append({
        "cmap_name": analog,
        "cells_used": int(sub["cell_id"].nunique()),
        "median_slope": med,
        "ci95_lo": lo,
        "ci95_hi": hi,
        "positive_fraction": frac_pos
    })

potency_table = (pd.DataFrame(rows2)
                 .sort_values("median_slope", ascending=False)
                 .reset_index(drop=True))
print("\nAnalog potency ranking by dose-slope (median across cells):")
print(potency_table.to_string(index=False))


### 1b) Quick visual — analog potency (median slope with 95% CI)

Bars show the **median slope** per analog; whiskers = bootstrap 95% CI.  
Higher bars → stronger increase of the core program with dose.


In [ ]:
# Bar plot with CIs
plt.figure(figsize=(7, 3.6))
order = potency_table.sort_values("median_slope", ascending=False)["cmap_name"]
ax = sns.pointplot(data=potency_table, x="cmap_name", y="median_slope",
                   order=order, join=False, errorbar=None)
# draw CI whiskers manually
for i, a in enumerate(order):
    row = potency_table.loc[potency_table["cmap_name"]==a].iloc[0]
    ax.vlines(i, row["ci95_lo"], row["ci95_hi"], linewidth=2)
plt.title("Analog potency by dose-slope (core_score ~ log10 dose)")
plt.xlabel("Analog"); plt.ylabel("Median slope across cells")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()


### 2) Statistical comparisons of core scores across groups

Non-parametric tests on **core_score**:
- Across cells **(Kruskal–Wallis)** + pairwise **Mann–Whitney** (BH-FDR).
- Across analogs **(Kruskal–Wallis)** + pairwise **Mann–Whitney** (BH-FDR).
We also report group medians to help interpret effect direction.


In [ ]:
# --- Kruskal–Wallis across cells and analogs; pairwise MW with BH-FDR ---

from itertools import combinations
from scipy.stats import kruskal, mannwhitneyu

def kw_and_pairwise(df, group_col):
    # Global KW
    groups = [v["core_score"].values for _, v in df.groupby(group_col)]
    kw_H, kw_p = kruskal(*groups)
    # Pairwise MW
    pairs, pvals = [], []
    for g1, g2 in combinations(sorted(df[group_col].dropna().unique()), 2):
        x = df.loc[df[group_col]==g1, "core_score"].values
        y = df.loc[df[group_col]==g2, "core_score"].values
        if len(x)>=5 and len(y)>=5:
            stat, p = mannwhitneyu(x, y, alternative="two-sided")
            pairs.append((g1, g2)); pvals.append(p)
    adj = np.full(len(pairs), np.nan)
    if pvals:
        adj = multipletests(pvals, method="fdr_bh")[1]
    pair_df = pd.DataFrame(pairs, columns=[f"{group_col}_1", f"{group_col}_2"])
    if len(pair_df):
        pair_df["pval"] = pvals; pair_df["fdr_bh"] = adj
    # Medians per group
    meds = (df.groupby(group_col)["core_score"]
              .median().rename("median_core_score").reset_index())
    return (kw_H, kw_p), pair_df.sort_values("fdr_bh"), meds.sort_values("median_core_score", ascending=False)

# Cells
(kwH_cell, kwp_cell), pw_cell, meds_cell = kw_and_pairwise(core_score_meta, "cell_id")
print(f"Cells — Kruskal–Wallis: H={kwH_cell:.2f}, p={kwp_cell:.3e}")
print("Top group medians (cells):")
print(meds_cell.head().to_string(index=False))
print("\nSignificant pairwise (BH q<0.05):")
print(pw_cell.query("fdr_bh < 0.05").head(10).to_string(index=False))

# Analogs
(kwH_an, kwp_an), pw_an, meds_an = kw_and_pairwise(core_score_meta, "cmap_name")
print(f"\nAnalogs — Kruskal–Wallis: H={kwH_an:.2f}, p={kwp_an:.3e}")
print("Top group medians (analogs):")
print(meds_an.head().to_string(index=False))
print("\nSignificant pairwise (BH q<0.05):")
print(pw_an.query("fdr_bh < 0.05").head(10).to_string(index=False))


### 3) Robustness check — alternate core size

Recompute the core score using a **slightly different core** (e.g., 35 UP / 30 DOWN) and verify:
- Correlation of signature-level scores (old vs. new).
- Stability of **potency ranking** (median slopes per analog).


In [ ]:
# --- Rebuild core with different sizes and compare ---

N_CORE_UP2, N_CORE_DN2 = 35, 30

# New core sets
alt_up_ids = set(consensus_up.sort_values(["votes_up","global_mean_z"], ascending=[False, False])
                 .head(N_CORE_UP2)["gene_id"].astype(str))
alt_dn_ids = set(consensus_down.sort_values(["votes_down","global_mean_z"], ascending=[False, True])
                 .head(N_CORE_DN2)["gene_id"].astype(str))

# Compute new core scores
genes_str = pd.Index(exp_matrix.index.astype(str))
i_up2 = genes_str.isin(alt_up_ids)
i_dn2 = genes_str.isin(alt_dn_ids)
up2 = exp_matrix.loc[i_up2].mean(axis=0)
dn2 = exp_matrix.loc[i_dn2].mean(axis=0)

alt_scores = pd.DataFrame({"sig_id": exp_matrix.columns.astype(str),
                           "core_score_alt": (up2 - dn2).values})

# Join with original
merged_scores = core_score_meta.merge(alt_scores, on="sig_id", how="left")
rho_all = merged_scores["core_score"].corr(merged_scores["core_score_alt"], method="spearman")
print(f"Spearman correlation (original vs. alt core) across signatures: {rho_all:.3f}")

# Refit slopes with alt scores
rows_alt = []
for (analog, cell), sub in merged_scores.dropna(subset=["core_score_alt","log_dose"]).groupby(["cmap_name","cell_id"]):
    if (len(sub) >= 4) and (sub["log_dose"].nunique() >= 2):
        slope = fit_slope_ols(sub.rename(columns={"core_score_alt":"core_score"}))
        rows_alt.append({"cmap_name": analog, "cell_id": cell, "slope_alt": slope})
slopes_alt = pd.DataFrame(rows_alt)

# Summarize per analog (alt)
pot_alt = (slopes_alt.groupby("cmap_name")["slope_alt"].median()
           .rename("median_slope_alt").reset_index())

# Merge with original potency table and correlate
pot_compare = potency_table.merge(pot_alt, on="cmap_name", how="inner")
rho_pot = pot_compare["median_slope"].corr(pot_compare["median_slope_alt"], method="spearman")
print(f"Spearman correlation of potency medians (orig vs. alt core): {rho_pot:.3f}")

pot_compare.sort_values("median_slope", ascending=False).head()
